# RAG-Based Customer Support Assistant with Gemini, ChromaDB, LangGraph, and HITL

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) customer support assistant. It loads the uploaded PDFs from `pdfs/`, chunks their text, stores Gemini embeddings in ChromaDB, retrieves relevant chunks for each query, generates grounded answers with Gemini, and uses LangGraph to route low-confidence or complex cases to Human-in-the-Loop (HITL) review.

## What this notebook demonstrates
- PDF knowledge-base ingestion
- Chunking with metadata preservation
- Gemini embeddings with ChromaDB vector storage
- Contextual answer generation using Gemini
- LangGraph workflow control
- Conditional routing based on intent and retrieval confidence
- CLI-style HITL escalation inside a notebook

## 1. Install dependencies

## 2. Imports and configuration

The notebook reads your Gemini API key from the `GEMINI_API_KEY` environment variable. If it is missing, it asks for the key securely at runtime.

In [1]:
import os
import shutil
from getpass import getpass
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, StateGraph

load_dotenv()

PDF_DIR = Path("pdfs")
CHROMA_PATH = Path("chroma_db")
COLLECTION_NAME = "rag_customer_support_gemini"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4
ESCALATION_THRESHOLD = 0.45

EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")
CHAT_MODEL = os.getenv("GEMINI_CHAT_MODEL", "gemini-2.5-flash")

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass("Enter your Gemini API key: ").strip()
    os.environ["GEMINI_API_KEY"] = api_key

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY is required before running the RAG pipeline.")

print(f"PDF directory: {PDF_DIR.resolve()}")
print(f"ChromaDB path: {CHROMA_PATH.resolve()}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Chat model: {CHAT_MODEL}")

Enter your Gemini API key:  ········


PDF directory: C:\Users\nites\OneDrive\Documents\videsh Stuff\Final_Rag_Project\pdfs
ChromaDB path: C:\Users\nites\OneDrive\Documents\videsh Stuff\Final_Rag_Project\chroma_db
Embedding model: gemini-embedding-001
Chat model: gemini-2.5-flash


## 3. Check uploaded PDFs

This notebook expects your uploaded PDFs in the `pdfs/` folder. The next cell lists the detected files.

In [2]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR.resolve()}")

print(f"Detected {len(pdf_files)} PDF file(s):")
for pdf in pdf_files:
    print(f"- {pdf.name} ({pdf.stat().st_size:,} bytes)")

Detected 3 PDF file(s):
- HLD_RAG_Customer_Support.pdf (10,712 bytes)
- LLD_RAG_Customer_Support.pdf (11,794 bytes)
- Technical_Documentation_RAG.pdf (16,337 bytes)


## 4. Load and chunk the PDFs

Each page is loaded with metadata. Chunks retain `source`, `page`, and `chunk_id` so final answers can cite where the information came from.

In [3]:
def load_pdf_pages(pdf_paths: List[Path]) -> List[Document]:
    pages: List[Document] = []
    for pdf_path in pdf_paths:
        loader = PyPDFLoader(str(pdf_path))
        loaded_pages = loader.load()
        for page_doc in loaded_pages:
            page_doc.metadata["source"] = pdf_path.name
            page_doc.metadata["page"] = int(page_doc.metadata.get("page", 0)) + 1
        pages.extend(loaded_pages)
    return pages


def split_documents(pages: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(pages)
    for idx, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = idx
    return chunks


pages = load_pdf_pages(pdf_files)
chunks = split_documents(pages)

print(f"Loaded {len(pages)} page(s).")
print(f"Created {len(chunks)} chunk(s).")
print("\nSample chunk metadata:")
print(chunks[0].metadata)
print("\nSample chunk text:")
print(chunks[0].page_content[:700])

Loaded 15 page(s).
Created 32 chunk(s).

Sample chunk metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-01T03:27:26+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T03:27:26+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'HLD_RAG_Customer_Support.pdf', 'total_pages': 4, 'page': 1, 'page_label': '1', 'chunk_id': 0}

Sample chunk text:
RAG-Based Customer Support
 Assistant
 HIGH-LEVEL DESIGN (HLD) DOCUMENT
 LangGraph · ChromaDB · Groq LLaMA-3 · HITL Escalation
 Innomatics Research Labs — Data Science & GenAI Internship
 February 2026 Cohort
1. System Overview
This document defines the High-Level Design of a Retrieval-Augmented Generation (RAG) system built for
customer support automation. The system ingests a PDF knowledge base, chunks and indexes it using
semantic embeddings in ChromaDB, and responds to natural language queries via a LangGraph
orchestrated

## 5. Create Gemini embeddings and store them in ChromaDB

This cell resets the local vector store and rebuilds it from the uploaded PDFs. Set `RESET_VECTOR_STORE = False` if you want to reuse an existing ChromaDB index.

In [4]:
RESET_VECTOR_STORE = True

if RESET_VECTOR_STORE and CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_PATH),
)

print(f"Stored {len(chunks)} chunk(s) in ChromaDB collection '{COLLECTION_NAME}'.")

Stored 32 chunk(s) in ChromaDB collection 'rag_customer_support_gemini'.


## 6. Retrieval and prompt helpers

The assistant only answers from retrieved context. If the context is weak or the intent is complex, the graph routes the response to HITL review.

In [5]:
llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    temperature=0.2,
)


def format_source(doc: Document) -> str:
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "?")
    chunk_id = doc.metadata.get("chunk_id", "?")
    return f"{source}, page {page}, chunk {chunk_id}"


def retrieve_context(query: str, top_k: int = TOP_K) -> Dict[str, Any]:
    results = vector_store.similarity_search_with_relevance_scores(query, k=top_k)
    retrieved = []
    for doc, score in results:
        retrieved.append({"doc": doc, "score": float(score), "source": format_source(doc)})
    confidence = max([item["score"] for item in retrieved], default=0.0)
    return {"retrieved": retrieved, "confidence": confidence}


def build_context_block(retrieved: List[Dict[str, Any]]) -> str:
    if not retrieved:
        return "No relevant context was retrieved."

    blocks = []
    for idx, item in enumerate(retrieved, start=1):
        doc = item["doc"]
        blocks.append(
            f"[Source {idx}: {item['source']} | score={item['score']:.3f}]\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n".join(blocks)


def detect_intent(query: str) -> str:
    lowered = query.lower()
    escalation_terms = [
        "refund dispute", "complaint", "angry", "legal", "lawsuit", "exception",
        "manager", "human", "escalate", "compensation", "policy exception",
        "medical", "security breach", "urgent", "delete my account",
    ]
    if any(term in lowered for term in escalation_terms):
        return "escalation_or_sensitive"
    if len(query.split()) > 45:
        return "complex_multi_part"
    return "standard_support_question"


def generate_answer(query: str, retrieved: List[Dict[str, Any]]) -> str:
    context_block = build_context_block(retrieved)
    prompt = f"""
You are a customer support assistant using Retrieval-Augmented Generation.
Answer the user's question using only the retrieved context below.

Rules:
1. If the context does not contain enough information, say that the available documents do not provide enough information.
2. Do not invent policies, prices, deadlines, or technical details.
3. Be concise, helpful, and professional.
4. Cite sources inline using the source labels, such as [Source 1].

Retrieved context:
{context_block}

User question:
{query}

Grounded answer:
""".strip()
    response = llm.invoke(prompt)
    return response.content


def should_escalate(intent: str, confidence: float, retrieved: List[Dict[str, Any]]) -> Optional[str]:
    if not retrieved:
        return "No relevant chunks were retrieved."
    if confidence < ESCALATION_THRESHOLD:
        return f"Retrieval confidence {confidence:.3f} is below threshold {ESCALATION_THRESHOLD:.2f}."
    if intent in {"escalation_or_sensitive", "complex_multi_part"}:
        return f"Query intent '{intent}' requires human review."
    return None

## 7. LangGraph workflow with conditional routing

The graph has a processing node and an output node. Conditional routing chooses whether to answer directly or escalate to the HITL path.

In [6]:
class RAGState(TypedDict, total=False):
    query: str
    intent: str
    retrieved_chunks: List[Dict[str, Any]]
    confidence: float
    route: Literal["direct", "hitl"]
    draft_answer: str
    final_answer: str
    sources: List[str]
    escalation_reason: Optional[str]
    human_reviewed: bool


def processing_node(state: RAGState) -> RAGState:
    query = state["query"]
    intent = detect_intent(query)
    retrieval = retrieve_context(query)
    retrieved = retrieval["retrieved"]
    confidence = retrieval["confidence"]
    draft_answer = generate_answer(query, retrieved) if retrieved else "I could not find relevant context in the uploaded PDFs."
    escalation_reason = should_escalate(intent, confidence, retrieved)
    route = "hitl" if escalation_reason else "direct"

    return {
        **state,
        "intent": intent,
        "retrieved_chunks": retrieved,
        "confidence": confidence,
        "route": route,
        "draft_answer": draft_answer,
        "sources": [item["source"] for item in retrieved],
        "escalation_reason": escalation_reason,
    }


def hitl_node(state: RAGState) -> RAGState:
    print("\n--- HUMAN-IN-THE-LOOP REVIEW REQUIRED ---")
    print(f"Query: {state['query']}")
    print(f"Intent: {state.get('intent')}")
    print(f"Confidence: {state.get('confidence', 0):.3f}")
    print(f"Escalation reason: {state.get('escalation_reason')}")
    print("\nRetrieved sources:")
    for source in state.get("sources", []):
        print(f"- {source}")
    print("\nDraft answer:")
    print(state.get("draft_answer", ""))

    decision = input("\nType 'approve' to use the draft, 'edit' to revise it, or anything else to replace it: ").strip().lower()
    if decision == "approve":
        final_answer = state.get("draft_answer", "")
    elif decision == "edit":
        human_text = input("Enter the revised answer: ").strip()
        final_answer = human_text or state.get("draft_answer", "")
    else:
        final_answer = input("Enter the final human response: ").strip()

    return {**state, "final_answer": final_answer, "human_reviewed": True}


def output_node(state: RAGState) -> RAGState:
    if not state.get("final_answer"):
        state["final_answer"] = state.get("draft_answer", "")
    state["human_reviewed"] = bool(state.get("human_reviewed", False))
    return state


def route_after_processing(state: RAGState) -> str:
    return "hitl" if state.get("route") == "hitl" else "output"


workflow = StateGraph(RAGState)
workflow.add_node("processing", processing_node)
workflow.add_node("hitl", hitl_node)
workflow.add_node("output", output_node)

workflow.set_entry_point("processing")
workflow.add_conditional_edges("processing", route_after_processing, {"hitl": "hitl", "output": "output"})
workflow.add_edge("hitl", "output")
workflow.add_edge("output", END)

rag_graph = workflow.compile()
print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## 8. Assistant runner

Use `ask_assistant()` for a single query. It returns the final answer, route, confidence score, and sources.

In [7]:
def ask_assistant(query: str, show_sources: bool = True) -> RAGState:
    result = rag_graph.invoke({"query": query})

    reviewed_label = "Human-reviewed" if result.get("human_reviewed") else "Automated"
    display(Markdown(f"### Answer ({reviewed_label})\n\n{result.get('final_answer', '')}"))
    print(f"Route: {result.get('route')}")
    print(f"Intent: {result.get('intent')}")
    print(f"Confidence: {result.get('confidence', 0):.3f}")
    if result.get("escalation_reason"):
        print(f"Escalation reason: {result.get('escalation_reason')}")

    if show_sources:
        print("\nSources:")
        for source in result.get("sources", []):
            print(f"- {source}")

    return result

## 9. Demo query: normal RAG answer

This question should be answerable from the uploaded RAG project PDFs.

In [8]:
normal_result = ask_assistant("What are the main components of the RAG customer support assistant architecture?")

### Answer (Automated)

The main components of the RAG customer support assistant architecture include:

*   **Document Ingestion Pipeline**: This pipeline consists of a PDF Loader, Text Splitter, Embedding Model, and ChromaDB Vector Store [Source 4].
*   **LangGraph Workflow Engine**: This engine orchestrates the workflow and includes components such as a Query Processor, Retriever, Router, and Answer Generator/HITL Escalator [Source 4].
*   **Groq LLaMA-3 (LLM)**: This LLM is used for generating responses [Source 3, Source 4].
*   **Human-in-the-Loop (HITL) Escalation**: This mechanism is used when the system cannot answer with sufficient confidence, routing complex cases to a human agent [Source 1, Source 3, Source 4].

The system also utilizes semantic retrieval (ChromaDB), intelligent workflow orchestration (LangGraph), and state-of-the-art LLM inference (Groq/LLaMA-3) [Source 2].

Route: direct
Intent: standard_support_question
Confidence: 0.731

Sources:
- Technical_Documentation_RAG.pdf, page 1, chunk 18
- Technical_Documentation_RAG.pdf, page 6, chunk 31
- HLD_RAG_Customer_Support.pdf, page 1, chunk 0
- HLD_RAG_Customer_Support.pdf, page 2, chunk 2


## 10. Demo query: low-confidence / missing-context route

This type of question should route to HITL if the retrieved context is weak or unrelated.

In [9]:
low_confidence_result = ask_assistant("What is the refund deadline for a damaged refrigerator order placed last Tuesday?")


--- HUMAN-IN-THE-LOOP REVIEW REQUIRED ---
Query: What is the refund deadline for a damaged refrigerator order placed last Tuesday?
Intent: standard_support_question
Confidence: 0.398
Escalation reason: Retrieval confidence 0.398 is below threshold 0.45.

Retrieved sources:
- Technical_Documentation_RAG.pdf, page 5, chunk 27
- Technical_Documentation_RAG.pdf, page 5, chunk 29
- Technical_Documentation_RAG.pdf, page 5, chunk 28
- LLD_RAG_Customer_Support.pdf, page 5, chunk 16

Draft answer:
The available documents do not provide enough information about the refund deadline for a damaged refrigerator order.



Type 'approve' to use the draft, 'edit' to revise it, or anything else to replace it:  approved
Enter the final human response:  nothing


### Answer (Human-reviewed)

nothing

Route: hitl
Intent: standard_support_question
Confidence: 0.398
Escalation reason: Retrieval confidence 0.398 is below threshold 0.45.

Sources:
- Technical_Documentation_RAG.pdf, page 5, chunk 27
- Technical_Documentation_RAG.pdf, page 5, chunk 29
- Technical_Documentation_RAG.pdf, page 5, chunk 28
- LLD_RAG_Customer_Support.pdf, page 5, chunk 16


## 11. Demo query: complex or escalation-sensitive route

This query intentionally includes escalation language so the graph demonstrates HITL routing.

In [10]:
escalation_result = ask_assistant("A customer is angry and wants a policy exception, compensation, and a manager escalation. How should the assistant respond?")


--- HUMAN-IN-THE-LOOP REVIEW REQUIRED ---
Query: A customer is angry and wants a policy exception, compensation, and a manager escalation. How should the assistant respond?
Intent: escalation_or_sensitive
Confidence: 0.542
Escalation reason: Query intent 'escalation_or_sensitive' requires human review.

Retrieved sources:
- Technical_Documentation_RAG.pdf, page 5, chunk 29
- LLD_RAG_Customer_Support.pdf, page 4, chunk 13
- Technical_Documentation_RAG.pdf, page 5, chunk 27
- Technical_Documentation_RAG.pdf, page 6, chunk 30

Draft answer:
The assistant should escalate the query to a human agent, as the presence of escalation keywords such as "complaint" (implied by the customer's anger and request for compensation/exception) and "manager escalation" would trigger an escalation route [Source 2]. The system is designed to escalate if an escalation keyword is matched in the query [Source 2].



Type 'approve' to use the draft, 'edit' to revise it, or anything else to replace it:  approve


### Answer (Human-reviewed)

The assistant should escalate the query to a human agent, as the presence of escalation keywords such as "complaint" (implied by the customer's anger and request for compensation/exception) and "manager escalation" would trigger an escalation route [Source 2]. The system is designed to escalate if an escalation keyword is matched in the query [Source 2].

Route: hitl
Intent: escalation_or_sensitive
Confidence: 0.542
Escalation reason: Query intent 'escalation_or_sensitive' requires human review.

Sources:
- Technical_Documentation_RAG.pdf, page 5, chunk 29
- LLD_RAG_Customer_Support.pdf, page 4, chunk 13
- Technical_Documentation_RAG.pdf, page 5, chunk 27
- Technical_Documentation_RAG.pdf, page 6, chunk 30


## 12. Inspect retrieved chunks for any query

Use this helper to see exactly what the retriever sent to the LLM.

In [11]:
def show_retrieved_chunks(query: str, top_k: int = TOP_K) -> None:
    retrieval = retrieve_context(query, top_k=top_k)
    print(f"Query: {query}")
    print(f"Top confidence: {retrieval['confidence']:.3f}")
    for idx, item in enumerate(retrieval["retrieved"], start=1):
        doc = item["doc"]
        print("\n" + "=" * 90)
        print(f"Source {idx}: {item['source']} | score={item['score']:.3f}")
        print("-" * 90)
        print(doc.page_content[:1200])


show_retrieved_chunks("Explain the HITL escalation design in this project.")

Query: Explain the HITL escalation design in this project.
Top confidence: 0.686

Source 1: LLD_RAG_Customer_Support.pdf, page 3, chunk 12 | score=0.686
------------------------------------------------------------------------------------------
state.
hitl_escalator
Logs escalation. Simulates/invokes human response. Stores in
state['human_response'].
3.2 Edge Definitions
 Edge
 Condition
START → query_processor
Always
query_processor → retriever_node
Always
retriever_node → router
Always
router → answer_generator
If state['route'] == 'answer'
router → hitl_escalator
If state['route'] == 'escalate'
answer_generator → END
Always
hitl_escalator → END
Always
4. Conditional Routing Logic

Source 2: LLD_RAG_Customer_Support.pdf, page 4, chunk 13 | score=0.681
------------------------------------------------------------------------------------------
def route_query(state: GraphState) -> GraphState: CONFIDENCE_THRESHOLD = 0.4
ESCALATION_KEYWORDS = ["urgent", "complaint", "human", "agent", "frus

## 13. Interactive chat loop

Run this cell to ask multiple questions. Type `exit` to stop.

In [ ]:
while True:
    user_query = input("\nAsk a support question, or type 'exit': ").strip()
    if user_query.lower() in {"exit", "quit", "q"}:
        print("Goodbye.")
        break
    if not user_query:
        continue
    ask_assistant(user_query)